# DQN Warm-Start Colab Runner

Notebook to launch DQN warm-start runs from supervised V2 checkpoints (`best_model.pt`) for `grid_3x3` and `heavy_hex_19`.

In [3]:
# 1) Sync repo and branch (important for --init-model-path support)
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive')
REPO_DIR = ROOT / 'rl-quantum-circuit-routing'
# 1) Resync code from GitHub
%cd /content/drive/MyDrive/rl-quantum-circuit-routing
!git fetch origin
!git checkout elora-DQN
!git pull origin elora-DQN
!git log -1 --oneline
!python main.py --help | grep init-model-path


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/rl-quantum-circuit-routing
error: pathspec 'elora-DQN' did not match any file(s) known to git
From https://github.com/helloelora/rl-quantum-circuit-routing
 * branch            elora-DQN  -> FETCH_HEAD
Already up to date.
5fd95c4 (HEAD -> version-27) Add DQN warm-start CLI and Colab notebook; log supervised rollout results
               [--init-model-path INIT_MODEL_PATH] [--save-path SAVE_PATH]
  --init-model-path INIT_MODEL_PATH


In [5]:
# 2) Safer helper (prints stderr if subprocess fails)
import sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

def run_dqn_warmstart(topo: str, seed: int, steps: int, depth: int, max_steps_cap: int):
    run_name = f"dqn_warm_{topo}_seed{seed}"
    init_model = REPO_DIR / "results" / "sanity_checks_v2" / f"sanity_v2_{topo}_seed{seed}" / "best_model.pt"
    assert init_model.exists(), f"Missing init model: {init_model}"

    cmd = [
        sys.executable, "main.py",
        "--project-root", str(REPO_DIR),
        "--algo", "dqn",
        "--run-name", run_name,
        "--topologies", topo,
        "--circuit-depth", str(depth),
        "--total-timesteps", str(steps),
        "--rollout-steps", "4096",
        "--learning-rate", "2e-4",
        "--gamma", "0.99",
        "--dqn-replay-size", "200000",
        "--dqn-min-replay-size", "10000",
        "--dqn-batch-size", "256",
        "--dqn-target-update-interval-steps", "2000",
        "--dqn-epsilon-start", "0.6",
        "--dqn-epsilon-end", "0.05",
        "--dqn-epsilon-decay-steps", "250000",
        "--completion-bonus", "15",
        "--timeout-penalty", "-8",
        "--gate-reward-coeff", "1.0",
        "--step-penalty", "-0.05",
        "--reverse-swap-penalty", "-0.2",
        "--repeat-swap-penalty-coeff", "-0.03",
        "--repeat-swap-penalty-cap", "-1.0",
        "--no-progress-penalty-coeff", "-0.008",
        "--no-progress-penalty-cap", "-0.4",
        "--distance-reward-coeff-start", "0.03",
        "--distance-reward-coeff-end", "0.015",
        "--max-steps-per-two-qubit-gate", "7",
        "--max-steps-min", "40",
        "--max-steps-max", str(max_steps_cap),
        "--min-two-qubit-gates", "8",
        "--eval-interval-updates", "20",
        "--eval-circuits-per-topology", "24",
        "--eval-circuit-depth", str(depth),
        "--eval-min-two-qubit-gates", "8",
        "--trace-interval-updates", "20",
        "--trace-cases-per-topology", "2",
        "--trace-max-steps", "220",
        "--trace-alert-dom-threshold", "0.6",
        "--trace-alert-backtrack-threshold", "0.5",
        "--trace-alert-patience", "2",
        "--seed", str(seed),
        "--device", "auto",
        "--init-model-path", str(init_model),
    ]

    print("\nRunning:", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.returncode != 0:
        print("STDOUT:\n", p.stdout)
        print("STDERR:\n", p.stderr)
        raise RuntimeError(f"Command failed with return code {p.returncode}")
    print(p.stdout)


In [6]:
# 3) Run grid_3x3 warm-start (3 seeds)
for s in [42, 123, 999]:
    run_dqn_warmstart(topo='grid_3x3', seed=s, steps=500000, depth=12, max_steps_cap=320)


Running: /usr/bin/python3 main.py --project-root /content/drive/MyDrive/rl-quantum-circuit-routing --algo dqn --run-name dqn_warm_grid_3x3_seed42 --topologies grid_3x3 --circuit-depth 12 --total-timesteps 500000 --rollout-steps 4096 --learning-rate 2e-4 --gamma 0.99 --dqn-replay-size 200000 --dqn-min-replay-size 10000 --dqn-batch-size 256 --dqn-target-update-interval-steps 2000 --dqn-epsilon-start 0.6 --dqn-epsilon-end 0.05 --dqn-epsilon-decay-steps 250000 --completion-bonus 15 --timeout-penalty -8 --gate-reward-coeff 1.0 --step-penalty -0.05 --reverse-swap-penalty -0.2 --repeat-swap-penalty-coeff -0.03 --repeat-swap-penalty-cap -1.0 --no-progress-penalty-coeff -0.008 --no-progress-penalty-cap -0.4 --distance-reward-coeff-start 0.03 --distance-reward-coeff-end 0.015 --max-steps-per-two-qubit-gate 7 --max-steps-min 40 --max-steps-max 320 --min-two-qubit-gates 8 --eval-interval-updates 20 --eval-circuits-per-topology 24 --eval-circuit-depth 12 --eval-min-two-qubit-gates 8 --trace-interv

In [7]:
# Compare grid_3x3 warm-start DQN runs (seed 42/123/999) directly from Drive
from pathlib import Path
import json, csv, math, statistics

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
SEEDS = [42, 123, 999]
TOPO = "grid_3x3"

def f3(x):
    return "nan" if x is None or (isinstance(x,float) and math.isnan(x)) else f"{x:.3f}"

rows = []
for s in SEEDS:
    run_dir = REPO_DIR / "runs" / f"dqn_warm_{TOPO}_seed{s}"
    stage_dir = run_dir / "single_stage"
    best_path = run_dir / "best_eval_metrics.json"
    metrics_path = stage_dir / "metrics.csv"

    row = {
        "seed": s,
        "run_dir": str(run_dir),
        "best_eval_improve": float("nan"),
        "best_eval_win": float("nan"),
        "best_eval_timeout": float("nan"),
        "best_eval_model_swaps": float("nan"),
        "best_eval_sabre_swaps": float("nan"),
        "last_eval_improve": float("nan"),
        "last_eval_win": float("nan"),
        "last_eval_timeout": float("nan"),
        "last_trace_dom": float("nan"),
        "last_trace_backtrack": float("nan"),
        "last_trace_timeout": float("nan"),
        "last_trace_alert_streak": float("nan"),
    }

    # best eval summary
    if best_path.exists():
        b = json.loads(best_path.read_text())
        m = b.get("metrics", {})
        row["best_eval_improve"] = float(m.get("eval_improvement_pct", float("nan")))
        row["best_eval_win"] = float(m.get("eval_win_rate", float("nan")))
        row["best_eval_timeout"] = float(m.get("eval_timeout_rate", float("nan")))
        row["best_eval_model_swaps"] = float(m.get("eval_mean_ppo_swaps", float("nan")))
        row["best_eval_sabre_swaps"] = float(m.get("eval_mean_sabre_swaps", float("nan")))

    # last available eval+trace from metrics.csv
    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        eval_rows = [r for r in data if r.get("eval_improvement_pct", "") not in ("", "nan", "None")]
        trace_rows = [r for r in data if r.get("trace_action_dom_ratio", "") not in ("", "nan", "None")]

        if eval_rows:
            last_eval = eval_rows[-1]
            row["last_eval_improve"] = float(last_eval["eval_improvement_pct"])
            row["last_eval_win"] = float(last_eval["eval_win_rate"])
            row["last_eval_timeout"] = float(last_eval["eval_timeout_rate"])

        if trace_rows:
            last_trace = trace_rows[-1]
            row["last_trace_dom"] = float(last_trace["trace_action_dom_ratio"])
            row["last_trace_backtrack"] = float(last_trace["trace_backtrack_rate"])
            row["last_trace_timeout"] = float(last_trace["trace_timeout_rate"])
            row["last_trace_alert_streak"] = float(last_trace["trace_alert_streak"])

    rows.append(row)

# print per-seed table
print(f"=== {TOPO} warm-start DQN comparison ===")
for r in rows:
    print(
        f"seed={r['seed']} | "
        f"best: improve={f3(r['best_eval_improve'])}% win={f3(r['best_eval_win'])} timeout={f3(r['best_eval_timeout'])} "
        f"swaps(model/sabre)={f3(r['best_eval_model_swaps'])}/{f3(r['best_eval_sabre_swaps'])} | "
        f"last_eval: improve={f3(r['last_eval_improve'])}% win={f3(r['last_eval_win'])} timeout={f3(r['last_eval_timeout'])} | "
        f"last_trace: dom={f3(r['last_trace_dom'])} back={f3(r['last_trace_backtrack'])} "
        f"t_timeout={f3(r['last_trace_timeout'])} streak={f3(r['last_trace_alert_streak'])}"
    )

# aggregate means/std
def agg(vals):
    vals = [v for v in vals if not (isinstance(v, float) and math.isnan(v))]
    if not vals:
        return float("nan"), float("nan")
    mean = statistics.mean(vals)
    std = statistics.pstdev(vals) if len(vals) > 1 else 0.0
    return mean, std

for key in [
    "best_eval_improve", "best_eval_win", "best_eval_timeout",
    "last_eval_improve", "last_eval_win", "last_eval_timeout",
    "last_trace_dom", "last_trace_backtrack", "last_trace_timeout"
]:
    m, s = agg([r[key] for r in rows])
    print(f"{key}: mean={f3(m)} std={f3(s)}")


=== grid_3x3 warm-start DQN comparison ===
seed=42 | best: improve=-1792.628% win=0.000 timeout=1.000 swaps(model/sabre)=243.250/13.500 | last_eval: improve=-1792.628% win=0.000 timeout=1.000 | last_trace: dom=0.975 back=0.968 t_timeout=1.000 streak=6.000
seed=123 | best: improve=-1704.522% win=0.000 timeout=1.000 swaps(model/sabre)=243.250/14.000 | last_eval: improve=-1704.522% win=0.000 timeout=1.000 | last_trace: dom=0.959 back=0.941 t_timeout=1.000 streak=6.000
seed=999 | best: improve=-1572.374% win=0.000 timeout=0.917 swaps(model/sabre)=225.042/13.583 | last_eval: improve=-1572.374% win=0.000 timeout=0.917 | last_trace: dom=0.980 back=0.970 t_timeout=1.000 streak=6.000
best_eval_improve: mean=-1689.842 std=90.516
best_eval_win: mean=0.000 std=0.000
best_eval_timeout: mean=0.972 std=0.039
last_eval_improve: mean=-1689.842 std=90.516
last_eval_win: mean=0.000 std=0.000
last_eval_timeout: mean=0.972 std=0.039
last_trace_dom: mean=0.971 std=0.009
last_trace_backtrack: mean=0.960 std=

In [8]:
# Quick gate before heavy training: evaluate the INIT supervised model on heavy_hex_19
from pathlib import Path
import sys, subprocess

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
model_path = REPO_DIR / "results" / "sanity_checks_v2" / "sanity_v2_heavy_hex_19_seed42" / "best_model.pt"
out_dir = REPO_DIR / "results" / "supervised_rollout_eval" / "heavy_probe_seed42_max450"

cmd = [
    sys.executable, "scripts/evaluate_supervised_policy_vs_sabre.py",
    "--project-root", str(REPO_DIR),
    "--model-path", str(model_path),
    "--output-dir", str(out_dir),
    "--topologies", "heavy_hex_19",
    "--circuits-per-topology", "120",
    "--circuit-depth", "14",
    "--max-steps", "450",
    "--min-two-qubit-gates", "8",
    "--seed", "42",
    "--device", "auto",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 scripts/evaluate_supervised_policy_vs_sabre.py --project-root /content/drive/MyDrive/rl-quantum-circuit-routing --model-path /content/drive/MyDrive/rl-quantum-circuit-routing/results/sanity_checks_v2/sanity_v2_heavy_hex_19_seed42/best_model.pt --output-dir /content/drive/MyDrive/rl-quantum-circuit-routing/results/supervised_rollout_eval/heavy_probe_seed42_max450 --topologies heavy_hex_19 --circuits-per-topology 120 --circuit-depth 14 --max-steps 450 --min-two-qubit-gates 8 --seed 42 --device auto


CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate_supervised_policy_vs_sabre.py', '--project-root', '/content/drive/MyDrive/rl-quantum-circuit-routing', '--model-path', '/content/drive/MyDrive/rl-quantum-circuit-routing/results/sanity_checks_v2/sanity_v2_heavy_hex_19_seed42/best_model.pt', '--output-dir', '/content/drive/MyDrive/rl-quantum-circuit-routing/results/supervised_rollout_eval/heavy_probe_seed42_max450', '--topologies', 'heavy_hex_19', '--circuits-per-topology', '120', '--circuit-depth', '14', '--max-steps', '450', '--min-two-qubit-gates', '8', '--seed', '42', '--device', 'auto'], returncode=0)

In [9]:
# Read heavy probe results (quick summary)
from pathlib import Path
import json, csv, statistics

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
OUT_DIR = REPO_DIR / "results" / "supervised_rollout_eval" / "heavy_probe_seed42_max450"

summary_path = OUT_DIR / "supervised_rollout_eval_summary.json"
csv_path = OUT_DIR / "supervised_rollout_eval.csv"

assert summary_path.exists(), f"Missing: {summary_path}"
assert csv_path.exists(), f"Missing: {csv_path}"

summary = json.loads(summary_path.read_text())
print("=== Summary JSON ===")
print(json.dumps(summary, indent=2))

# Extra breakdown from CSV
rows = []
with csv_path.open("r", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append({
            "model_swaps": float(r["model_swaps"]),
            "sabre_swaps": float(r["sabre_swaps"]),
            "improvement_pct": float(r["improvement_pct"]),
            "win_vs_sabre": int(r["win_vs_sabre"]),
            "timeout": int(r["timeout"]),
        })

impr = [r["improvement_pct"] for r in rows]
wins = [r["win_vs_sabre"] for r in rows]
tout = [r["timeout"] for r in rows]
m_sw = [r["model_swaps"] for r in rows]
s_sw = [r["sabre_swaps"] for r in rows]

print("\n=== CSV breakdown ===")
print("cases:", len(rows))
print("improvement mean/std:", round(statistics.mean(impr), 3), round(statistics.pstdev(impr), 3))
print("win_rate:", round(statistics.mean(wins), 3))
print("timeout_rate:", round(statistics.mean(tout), 3))
print("model_swaps mean:", round(statistics.mean(m_sw), 3))
print("sabre_swaps mean:", round(statistics.mean(s_sw), 3))

# Optional: count hard failures / wins
print("wins count:", sum(wins), "/", len(wins))
print("timeouts count:", sum(tout), "/", len(tout))


=== Summary JSON ===
{
  "model_path": "/content/drive/MyDrive/rl-quantum-circuit-routing/results/sanity_checks_v2/sanity_v2_heavy_hex_19_seed42/best_model.pt",
  "device": "cuda",
  "topologies": [
    "heavy_hex_19"
  ],
  "cases_total": 120,
  "mean_model_swaps": 450.0,
  "mean_sabre_swaps": 122.0,
  "mean_improvement_pct": -270.71533203125,
  "median_improvement_pct": -268.8524475097656,
  "win_rate_vs_sabre": 0.0,
  "timeout_rate": 1.0,
  "per_topology": {
    "heavy_hex_19": {
      "cases": 120,
      "mean_model_swaps": 450.0,
      "mean_sabre_swaps": 122.0,
      "mean_improvement_pct": -270.71533203125,
      "median_improvement_pct": -268.8524475097656,
      "win_rate_vs_sabre": 0.0,
      "timeout_rate": 1.0
    }
  }
}

=== CSV breakdown ===
cases: 120
improvement mean/std: -270.715 26.516
win_rate: 0
timeout_rate: 1
model_swaps mean: 450.0
sabre_swaps mean: 122.0
wins count: 0 / 120
timeouts count: 120 / 120


In [ ]:
run_dqn_warmstart(
    topo='heavy_hex_19',
    seed=42,
    steps=250000,
    depth=14,
    max_steps_cap=450
)

In [ ]:
# 4) Run heavy_hex_19 warm-start (3 seeds)
for s in [42, 123, 999]:
    run_dqn_warmstart(topo='heavy_hex_19', seed=s, steps=800000, depth=14, max_steps_cap=450)

In [ ]:
# 5) Quick summary from best_eval_metrics.json
import json
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

for topo in ['grid_3x3', 'heavy_hex_19']:
    for s in [42, 123, 999]:
        run = REPO_DIR / 'runs' / f"dqn_warm_{topo}_seed{s}" / 'best_eval_metrics.json'
        if run.exists():
            data = json.loads(run.read_text())
            m = data['metrics']
            print(
                f"{topo} seed={s} "
                f"improve={m['eval_improvement_pct']:.2f}% "
                f"win={m['eval_win_rate']:.3f} "
                f"timeout={m['eval_timeout_rate']:.3f} "
                f"ppo_swaps={m['eval_mean_ppo_swaps']:.2f} "
                f"sabre_swaps={m['eval_mean_sabre_swaps']:.2f}"
            )
        else:
            print(f"missing: {run}")

## Clean execution path (recommended)

These cells are a clean, ordered flow for DQN warm-start experiments.
Your draft/scratch cells are kept unchanged below.


In [4]:
%pip install -q --upgrade --force-reinstall --no-cache-dir -r /content/drive/MyDrive/rl-quantum-circuit-routing/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 32.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 322.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 96.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 113.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 424.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 412.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.7/181.7 kB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 297.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 300.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 89.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 105.3 MB/s eta 0:00:000

In [ ]:
# A) Setup + sync branch (robust)
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BRANCH = 'elora-DQN'
REPO_URL = 'https://github.com/helloelora/rl-quantum-circuit-routing.git'
REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

if not (REPO_DIR / '.git').exists():
    !git clone -b {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin --prune
    !git show-ref --verify --quiet refs/heads/{BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!git branch --show-current
!git log -1 --oneline


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1386/1759628817.py", line 14, in <cell line: 0>
    get_ipython().run_line_magic('cd', '{REPO_DIR}')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (mos

Exception ignored in atexit callback: <bound method InteractiveShell.atexit_operations of <google.colab._shell.Shell object at 0x7eddda4b9ca0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3924, in atexit_operations
    self.reset(new_session=False)
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 1445, in reset
    self.history_manager.reset(new_session)
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_history.py", line 33, in reset
    super(ColabHistoryManager, self).reset(new_session=new_session)
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/history.py", line 592, in reset
    self.dir_hist[:] = [os.getcwd()]
                        ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected


In [5]:
# B) Stable helper to launch one warm-start DQN run
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

def run_dqn_warmstart(
    topo: str,
    seed: int,
    steps: int,
    depth: int,
    max_steps_cap: int,
    eval_cases: int = 24,
    trace_cases: int = 2,
    tag: str = 'warm',
):
    run_name = f'dqn_{tag}_{topo}_seed{seed}'
    init_model = REPO_DIR / 'results' / 'sanity_checks_v2' / f'sanity_v2_{topo}_seed{seed}' / 'best_model.pt'
    if not init_model.exists():
        raise FileNotFoundError(f'Missing init model: {init_model}')

    cmd = [
        sys.executable, 'main.py',
        '--project-root', str(REPO_DIR),
        '--algo', 'dqn',
        '--run-name', run_name,
        '--topologies', topo,
        '--circuit-depth', str(depth),
        '--total-timesteps', str(steps),
        '--rollout-steps', '4096',
        '--learning-rate', '2e-4',
        '--gamma', '0.99',
        '--dqn-replay-size', '200000',
        '--dqn-min-replay-size', '10000',
        '--dqn-batch-size', '256',
        '--dqn-target-update-interval-steps', '2000',
        '--dqn-epsilon-start', '0.6',
        '--dqn-epsilon-end', '0.05',
        '--dqn-epsilon-decay-steps', str(max(steps // 2, 200000)),
        '--completion-bonus', '15',
        '--timeout-penalty', '-8',
        '--gate-reward-coeff', '1.0',
        '--step-penalty', '-0.05',
        '--reverse-swap-penalty', '-0.2',
        '--repeat-swap-penalty-coeff', '-0.03',
        '--repeat-swap-penalty-cap', '-1.0',
        '--no-progress-penalty-coeff', '-0.008',
        '--no-progress-penalty-cap', '-0.4',
        '--distance-reward-coeff-start', '0.03',
        '--distance-reward-coeff-end', '0.015',
        '--max-steps-per-two-qubit-gate', '7',
        '--max-steps-min', '40',
        '--max-steps-max', str(max_steps_cap),
        '--min-two-qubit-gates', '8',
        '--eval-interval-updates', '20',
        '--eval-circuits-per-topology', str(eval_cases),
        '--eval-circuit-depth', str(depth),
        '--eval-min-two-qubit-gates', '8',
        '--trace-interval-updates', '20',
        '--trace-cases-per-topology', str(trace_cases),
        '--trace-max-steps', '220',
        '--trace-alert-dom-threshold', '0.6',
        '--trace-alert-backtrack-threshold', '0.5',
        '--trace-alert-patience', '2',
        '--seed', str(seed),
        '--device', 'auto',
        '--init-model-path', str(init_model),
    ]

    print('Running:', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
    if p.returncode != 0:
        print('STDOUT:\n', p.stdout)
        print('STDERR:\n', p.stderr)
        raise RuntimeError(f'Run failed with return code {p.returncode}')

    print(p.stdout[-4000:])
    return run_name


In [6]:
# C) Recommended experiment blocks (pick what you need)
# 1) Grid, full 3 seeds (long):
# for s in [42, 123, 999]:
#     run_dqn_warmstart(topo='grid_3x3', seed=s, steps=500000, depth=12, max_steps_cap=320, eval_cases=24, trace_cases=2, tag='warm_grid')

# 2) Heavy quick probe, single seed (recommended before long heavy):
run_dqn_warmstart(topo='heavy_hex_19', seed=42, steps=250000, depth=14, max_steps_cap=450, eval_cases=24, trace_cases=2, tag='probe_heavy')

# 3) Heavy full 3 seeds (very long):
# for s in [42, 123, 999]:
#     run_dqn_warmstart(topo='heavy_hex_19', seed=s, steps=800000, depth=14, max_steps_cap=450, eval_cases=24, trace_cases=2, tag='warm_heavy')


Running: /usr/bin/python3 main.py --project-root /content/drive/MyDrive/rl-quantum-circuit-routing --algo dqn --run-name dqn_probe_heavy_heavy_hex_19_seed42 --topologies heavy_hex_19 --circuit-depth 14 --total-timesteps 250000 --rollout-steps 4096 --learning-rate 2e-4 --gamma 0.99 --dqn-replay-size 200000 --dqn-min-replay-size 10000 --dqn-batch-size 256 --dqn-target-update-interval-steps 2000 --dqn-epsilon-start 0.6 --dqn-epsilon-end 0.05 --dqn-epsilon-decay-steps 200000 --completion-bonus 15 --timeout-penalty -8 --gate-reward-coeff 1.0 --step-penalty -0.05 --reverse-swap-penalty -0.2 --repeat-swap-penalty-coeff -0.03 --repeat-swap-penalty-cap -1.0 --no-progress-penalty-coeff -0.008 --no-progress-penalty-cap -0.4 --distance-reward-coeff-start 0.03 --distance-reward-coeff-end 0.015 --max-steps-per-two-qubit-gate 7 --max-steps-min 40 --max-steps-max 450 --min-two-qubit-gates 8 --eval-interval-updates 20 --eval-circuits-per-topology 24 --eval-circuit-depth 14 --eval-min-two-qubit-gates 8 

'dqn_probe_heavy_heavy_hex_19_seed42'

In [8]:
# D) Compare runs quickly (no file download needed)
import csv, json, math, statistics
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')

def _f3(x):
    if x is None:
        return 'nan'
    try:
        v = float(x)
        if math.isnan(v):
            return 'nan'
        return f'{v:.3f}'
    except Exception:
        return str(x)

def compare_runs(run_names):
    rows = []
    for rn in run_names:
        run_dir = REPO_DIR / 'runs' / rn
        best_path = run_dir / 'best_eval_metrics.json'

        stage_dir = run_dir / 'single_stage'
        if not stage_dir.exists():
            stage_dirs = sorted([p for p in run_dir.glob('stage*') if p.is_dir()])
            stage_dir = stage_dirs[-1] if stage_dirs else run_dir

        metrics_path = stage_dir / 'metrics.csv'

        row = {
            'run': rn,
            'best_improve': float('nan'),
            'best_win': float('nan'),
            'best_timeout': float('nan'),
            'best_model_swaps': float('nan'),
            'best_sabre_swaps': float('nan'),
            'last_eval_improve': float('nan'),
            'last_eval_win': float('nan'),
            'last_eval_timeout': float('nan'),
            'last_trace_dom': float('nan'),
            'last_trace_back': float('nan'),
            'last_trace_timeout': float('nan'),
            'last_trace_streak': float('nan'),
        }

        if best_path.exists():
            b = json.loads(best_path.read_text())
            m = b.get('metrics', {})
            row['best_improve'] = float(m.get('eval_improvement_pct', float('nan')))
            row['best_win'] = float(m.get('eval_win_rate', float('nan')))
            row['best_timeout'] = float(m.get('eval_timeout_rate', float('nan')))
            row['best_model_swaps'] = float(m.get('eval_mean_ppo_swaps', float('nan')))
            row['best_sabre_swaps'] = float(m.get('eval_mean_sabre_swaps', float('nan')))

        if metrics_path.exists():
            with metrics_path.open('r', encoding='utf-8') as f:
                data = list(csv.DictReader(f))

            eval_rows = [r for r in data if r.get('eval_improvement_pct', '') not in ('', 'nan', 'None')]
            trace_rows = [r for r in data if r.get('trace_action_dom_ratio', '') not in ('', 'nan', 'None')]

            if eval_rows:
                e = eval_rows[-1]
                row['last_eval_improve'] = float(e['eval_improvement_pct'])
                row['last_eval_win'] = float(e['eval_win_rate'])
                row['last_eval_timeout'] = float(e['eval_timeout_rate'])

            if trace_rows:
                t = trace_rows[-1]
                row['last_trace_dom'] = float(t['trace_action_dom_ratio'])
                row['last_trace_back'] = float(t['trace_backtrack_rate'])
                row['last_trace_timeout'] = float(t['trace_timeout_rate'])
                row['last_trace_streak'] = float(t.get('trace_alert_streak', 'nan'))

        rows.append(row)

    print('=== Warm-start DQN comparison ===')
    for r in rows:
        print(
            f"{r['run']} | best: imp={_f3(r['best_improve'])}% win={_f3(r['best_win'])} timeout={_f3(r['best_timeout'])} "
            f"swaps={_f3(r['best_model_swaps'])}/{_f3(r['best_sabre_swaps'])} | "
            f"last_eval: imp={_f3(r['last_eval_improve'])}% win={_f3(r['last_eval_win'])} timeout={_f3(r['last_eval_timeout'])} | "
            f"trace: dom={_f3(r['last_trace_dom'])} back={_f3(r['last_trace_back'])} t_timeout={_f3(r['last_trace_timeout'])} streak={_f3(r['last_trace_streak'])}"
        )

    for k in ['best_improve', 'best_win', 'best_timeout', 'last_eval_improve', 'last_eval_win', 'last_eval_timeout', 'last_trace_dom', 'last_trace_back']:
        vals = [r[k] for r in rows if not (isinstance(r[k], float) and math.isnan(r[k]))]
        if vals:
            std = statistics.pstdev(vals) if len(vals) > 1 else 0.0
            print(f"{k}: mean={statistics.mean(vals):.3f} std={std:.3f}")

# Example:
compare_runs(['dqn_probe_heavy_heavy_hex_19_seed42'])


=== Warm-start DQN comparison ===
dqn_probe_heavy_heavy_hex_19_seed42 | best: imp=-271.620% win=0.000 timeout=1.000 swaps=450.000/121.625 | last_eval: imp=-271.620% win=0.000 timeout=1.000 | trace: dom=0.989 back=0.984 t_timeout=1.000 streak=3.000
best_improve: mean=-271.620 std=0.000
best_win: mean=0.000 std=0.000
best_timeout: mean=1.000 std=0.000
last_eval_improve: mean=-271.620 std=0.000
last_eval_win: mean=0.000 std=0.000
last_eval_timeout: mean=1.000 std=0.000
last_trace_dom: mean=0.989 std=0.000
last_trace_back: mean=0.984 std=0.000


In [9]:
# Run de stabilisation multi-topologies (linear + grid), sans heavy
from datetime import datetime
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/rl-quantum-circuit-routing')
RUN_NAME = "dqn_stab_lingrid_" + datetime.now().strftime("%Y%m%d_%H%M%S")

%cd {REPO_DIR}
!python main.py \
  --project-root /content/drive/MyDrive/rl-quantum-circuit-routing \
  --algo dqn \
  --run-name {RUN_NAME} \
  --topologies linear_5,grid_3x3 \
  --circuit-depth 12 \
  --total-timesteps 350000 \
  --rollout-steps 4096 \
  --learning-rate 2e-4 \
  --gamma 0.99 \
  --dqn-replay-size 200000 \
  --dqn-min-replay-size 10000 \
  --dqn-batch-size 256 \
  --dqn-target-update-interval-steps 2000 \
  --dqn-epsilon-start 0.8 \
  --dqn-epsilon-end 0.08 \
  --dqn-epsilon-decay-steps 280000 \
  --completion-bonus 15 \
  --timeout-penalty -8 \
  --gate-reward-coeff 1.0 \
  --step-penalty -0.05 \
  --reverse-swap-penalty -0.2 \
  --repeat-swap-penalty-coeff -0.05 \
  --repeat-swap-penalty-cap -1.2 \
  --no-progress-penalty-coeff -0.02 \
  --no-progress-penalty-cap -0.8 \
  --distance-reward-coeff-start 0.03 \
  --distance-reward-coeff-end 0.015 \
  --max-steps-per-two-qubit-gate 7 \
  --max-steps-min 40 \
  --max-steps-max 260 \
  --min-two-qubit-gates 8 \
  --eval-interval-updates 20 \
  --eval-circuits-per-topology 24 \
  --eval-circuit-depth 12 \
  --eval-min-two-qubit-gates 8 \
  --trace-interval-updates 20 \
  --trace-cases-per-topology 2 \
  --trace-max-steps 200 \
  --trace-alert-dom-threshold 0.6 \
  --trace-alert-backtrack-threshold 0.5 \
  --trace-alert-patience 2 \
  --seed 42 \
  --device auto

print("RUN_NAME =", RUN_NAME)


/content/drive/MyDrive/rl-quantum-circuit-routing
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Starting DQN training with settings:
  PROJECT_ROOT=/content/drive/MyDrive/rl-quantum-circuit-routing
  DATA_ROOT=/content/drive/MyDrive/rl-quantum-circuit-routing/data
  RUN_ROOT=/content/drive/MyDrive/rl-quantum-circuit-routing/runs
  RUN_DIR=/content/drive/MyDrive/rl-quantum-circuit-routing/runs/dqn_stab_lingrid_20260324_091230
  topologies=['linear_5', 'grid_3x3'] (stage3 if curriculum)
  initial_mapping_strategy=mixed (80% random, 20% SABRE)
  strategy_masking=off (topology validity mask only)
  gamma_decay=0.5
  completion_bonus=15.0
  timeout_penalty=-8.0
  gate_reward_coef

In [10]:
# Analyse rapide du run (utilise ta fonction compare_runs déjà définie)
compare_runs([RUN_NAME])

=== Warm-start DQN comparison ===
dqn_stab_lingrid_20260324_091230 | best: imp=-1342.975% win=0.021 timeout=0.896 swaps=165.396/11.792 | last_eval: imp=-1502.864% win=0.000 timeout=1.000 | trace: dom=0.977 back=0.971 t_timeout=1.000 streak=4.000
best_improve: mean=-1342.975 std=0.000
best_win: mean=0.021 std=0.000
best_timeout: mean=0.896 std=0.000
last_eval_improve: mean=-1502.864 std=0.000
last_eval_win: mean=0.000 std=0.000
last_eval_timeout: mean=1.000 std=0.000
last_trace_dom: mean=0.977 std=0.000
last_trace_back: mean=0.971 std=0.000
